# matched-planet counts per catalog and matching method
- horizontal bars: NEA vs Exo-MerCat, split by ID-only / coordinates-only / both
- exports to `report/figures/crossmatch_yield.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from collections import Counter

from report_plots import common, data


In [ ]:
# NB: the original notebook passed input_epoch=2000 to the Crossmatcher
# constructor, where it was silently swallowed by **kwargs -- the committed
# figure therefore used the flat default search radius, reproduced here by
# not passing input_epoch anywhere. minimum_search_radius is kept for
# faithfulness but is inert without an input epoch.
cm, input_table, result = data.nea_crossmatch(minimum_search_radius=1*u.arcsec)
nea_matches = Counter(result["match_type"])

cme, _, emc_result = data.emc_crossmatch(dedup_input=False)
emc_matches = Counter(emc_result["match_type"])

nea_matches, emc_matches


In [ ]:
catalog_names = ["NASA E.A.", "Exo-MerCat"]
match_types = ["id", "coordinates", "id+coordinates"]
match_labels = ["Only ID", "Only Coordinates", "Matched by both"]
values = np.array([
    [nea_matches.get(match_type, 0) for match_type in match_types],
    [emc_matches.get(match_type, 0) for match_type in match_types],
])

x = np.arange(len(catalog_names))
height = 0.25

fig, ax = plt.subplots(figsize=(4.85, 1.5))

colors = {
    "id": "#D4D4D4",
    "coordinates": "#F2D3A2",
    "id+coordinates": "#C0DCEB",
}

bars = []
for i, match_type in enumerate(match_types):
    for j, catalog_name in enumerate(catalog_names):
        bars.append(
            ax.barh(
                x[j] + (i - 1) * height,
                values[j, i],
                height,
                color=colors[match_type],
                edgecolor="black",
                linewidth=0.9,
            )
        )

ax.set_yticks(x)
ax.set_yticklabels(catalog_names)
ax.invert_yaxis()
ax.set_xlabel("Matched planets")
ax.grid(axis="x", linestyle="--", linewidth=0.7, alpha=0.35)
ax.set_axisbelow(True)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=colors[match_type], edgecolor="black", linewidth=0.9, label=label)
    for match_type, label in zip(match_types, match_labels)
]
ax.legend(
    legend_handles,
    match_labels,
    frameon=True,
    fancybox=True,
    fontsize=8,
)

for bars_group in bars:
    for bar in bars_group:
        width = int(bar.get_width())
        innie = width > 200
        ax.text(
            width + (10 if not innie else -20),
            bar.get_y() + bar.get_height() / 2,
            f"{width}",
            va="center",
            ha="left" if not innie else "right",
            fontsize=7,
            color="black",
        )

ax.set_xlim(0, max(values.max(), 1) * 1.02)
plt.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

common.save_figure("crossmatch_yield")
